[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/rotation.ipynb)

# Rotational motion full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the video's "Problems, step by step" chapters) and **tier 3** stretch problems
(run it backwards, combine two ideas, spot the trap). Each tier is harder than the last.

Between the tiers are the course's **simulations** to play with: burst the firework, pull the puck's cord, switch off
the fan, and drop the bucket down the well. Each one steps Newton's laws forward a tiny slice of time at a time, and
checks itself against the formula from the video. Under every tier there's also a short **how you'd do this in
Python** cell, with every answer of that tier worked out in code.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no
install) and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Two decimals are
enough, in the units the box names; g = 9.8 m/s². You can type the arithmetic too: `2*pi*600/60`, `sqrt(2*30/0.125)`,
and a unit at the end is fine: `0.6 m`. For several answers, separate them with commas, in the order the question
lists them (the box's label shows the order). You can also put your answer between the quotes in a cell's last line,
`practice("P1.1", "")`, and run the cell: it checks straight away.

**The cheat sheet.** Every formula and table from the course on two pages:
[rotation cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/rotation_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the simulator and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import interact, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # bodies and paths in chalk, chalk smudge, ghost lines
SKY, VIOLET, CORAL, GOLD, MINT = "#6fb6bb", "#b49ad6", "#e3896b", "#e8c56b", "#a0c27c"
# bodies and paths CHALK · velocity SKY · acceleration MINT · force CORAL · energy VIOLET · the one thing to look at GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

G = 9.8                                               # m/s², the course's g

# ---- the simulator: the same one the video's moving pictures come from ----
def simulate(deriv, state, t_end, dt=1e-3, until=None):
    """Fixed-step fourth-order Runge–Kutta. deriv(t, s) gives ds/dt; state is a tuple of numbers. Returns the list of
    (t, state) samples from 0 to t_end, or, with until = (i, level), up to the moment s[i] falls below level (the last
    sample placed on that moment by straight-line interpolation: a landing, a stop)."""
    t, s = 0.0, tuple(float(v) for v in state)
    out = [(t, s)]
    for _ in range(int(round(t_end / dt))):
        k1 = deriv(t, s)
        k2 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k1)))
        k3 = deriv(t + dt / 2, tuple(a + dt / 2 * b for a, b in zip(s, k2)))
        k4 = deriv(t + dt, tuple(a + dt * b for a, b in zip(s, k3)))
        s2 = tuple(a + dt / 6 * (b1 + 2 * b2 + 2 * b3 + b4) for a, b1, b2, b3, b4 in zip(s, k1, k2, k3, k4))
        if until is not None and s2[until[0]] < until[1]:
            i, level = until
            f = (level - s[i]) / (s2[i] - s[i])
            out.append((t + f * dt, tuple(a + f * (b - a) for a, b in zip(s, s2))))
            return out
        t, s = t + dt, s2
        out.append((t, s))
    return out

def close(a, b, rel=1e-4):
    """True when a simulated value matches the formula (to one part in ten thousand by default)."""
    return abs(a - b) <= max(rel * max(abs(a), abs(b)), 1e-9)

# the simulator checks itself: a ball thrown at 20 m/s and 30° lands v² sin 2θ ÷ g away
_run = simulate(lambda t, s: (s[2], s[3], 0.0, -G), (0, 0, 20 * math.cos(math.pi / 6), 20 * math.sin(math.pi / 6)), 10,
                1e-3, until=(1, 0.0))
assert close(_run[-1][1][0], 400 * math.sin(math.pi / 3) / G) and close(_run[-1][0], 20 / G)

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}

def calc(text):
    """Plain arithmetic only: numbers, + − × ÷, powers, brackets, pi and sqrt(…)."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id == "pi": return math.pi
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") == "sqrt" and len(n.args) == 1: return math.sqrt(ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = re.compile(r"\s*(kg ?m²/s|kg ?m\*\*2/s|kg ?m²|kg ?m\*\*2|n ?m ?s|n ?m|rad/s²|rad/s\*\*2|rad/s|rad|rpm|turns?|"
                  r"kw|kj|mj|km|kg|m/s²|m/s\*\*2|m/s|°|j|w|n|m|s)\s*$", re.I)

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**").replace("π", "pi")
    try: return calc(UNIT.sub("", s).strip() or s)
    except Exception: return None

def right(got, want, tol):
    return got is not None and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Centre of mass',
     "q": 'A 4 kg mass sits at x = 0 and a 1 kg mass at x = 1.0 m on a light rod. Where is their centre of mass (m)?',
     "parts": ['X (m)'],
     "answer": [0.2], "tol": [0.005],
     "hint": 'X = Σ m x ÷ Σ m',
     "working": 'X = (4 × 0 + 1 × 1.0)/(4 + 1) = 0.2 m: near the heavier mass'},
    {"id": 'P1.2', "tier": 1, "group": 'Centre of mass',
     "q": 'A uniform plank is 2.4 m long. How far from one end is its centre of mass (m)?',
     "parts": ['distance from one end (m)'],
     "answer": [1.2], "tol": [0.005],
     "hint": 'uniform: the middle',
     "working": 'Uniform: by symmetry, the middle, 2.4/2 = 1.2 m'},
    {"id": 'P1.3', "tier": 1, "group": 'The cross product and angular velocity',
     "q": 'Two vectors of size 6 and 2 make an angle of 30°. What is the size of their cross product?',
     "parts": ['|A × B|'],
     "answer": [5.999999999999999], "tol": [0.005],
     "hint": '|A × B| = A B sin θ',
     "working": '|A × B| = A B sin θ = 6 × 2 × 0.5 = 6'},
    {"id": 'P1.4', "tier": 1, "group": 'The cross product and angular velocity',
     "q": 'A drill turns at 600 revolutions per minute. What is its angular velocity in rad/s?',
     "parts": ['ω (rad/s)'],
     "answer": [62.83185307179586], "tol": [0.005],
     "hint": 'ω = 2π × rpm ÷ 60',
     "working": '600 × 2π/60 = 20π = 62.83 rad/s'},
    {"id": 'P1.5', "tier": 1, "group": 'The cross product and angular velocity',
     "q": 'A point 0.4 m from the axis of a wheel turning at 10 rad/s: what is its speed (m/s)?',
     "parts": ['v (m/s)'],
     "answer": [4.0], "tol": [0.005],
     "hint": 'v = ω r',
     "working": 'v = ω r = 10 × 0.4 = 4 m/s'},
    {"id": 'P1.6', "tier": 1, "group": 'Torque and angular momentum',
     "q": 'You push with 50 N at right angles to a spanner, 0.3 m from the nut. What torque do you apply (N m)?',
     "parts": ['τ (N m)'],
     "answer": [15.0], "tol": [0.005],
     "hint": 'τ = r F sin θ',
     "working": 'τ = r F sin 90° = 0.3 × 50 = 15 N m'},
    {"id": 'P1.7', "tier": 1, "group": 'Equilibrium',
     "q": "A 20 kg child sits 2 m from a see-saw's pivot. What mass, 1 m from the pivot on the other side, balances it (kg)?",
     "parts": ['mass (kg)'],
     "answer": [40.0], "tol": [0.005],
     "hint": 'm₁ d₁ = m₂ d₂',
     "working": "20 × 2 = m × 1 → m = 40 kg (the weights' g cancels)"},
    {"id": 'P1.8', "tier": 1, "group": 'Moment of inertia',
     "q": 'What is the moment of inertia of a 3 kg mass, small enough to treat as a point, 0.5 m from an axis (kg m²)?',
     "parts": ['I (kg m²)'],
     "answer": [0.75], "tol": [0.005],
     "hint": 'I = m r²',
     "working": 'I = m r² = 3 × 0.25 = 0.75 kg m²'},
    {"id": 'P1.9', "tier": 1, "group": 'Rotational kinematics',
     "q": 'A wheel turning at 2 rad/s speeds up at 3 rad/s² for 4 s. What is its angular velocity then (rad/s)?',
     "parts": ['ω (rad/s)'],
     "answer": [14.0], "tol": [0.005],
     "hint": 'ω = ω₀ + α t',
     "working": 'ω = ω₀ + α t = 2 + 3 × 4 = 14 rad/s'},
    {"id": 'P1.10', "tier": 1, "group": 'Rotational dynamics',
     "q": 'A motor gives a torque of 50 N m while turning at 100 rad/s. What power does it deliver (W)?',
     "parts": ['P (W)'],
     "answer": [5000.0], "tol": [0.005],
     "hint": 'P = τ ω',
     "working": 'P = τ ω = 50 × 100 = 5,000 W'},
    {"id": 'P2.1', "tier": 2, "group": 'Centre of mass',
     "q": 'Masses of 2 kg at (0, 0), 3 kg at (4, 0) and 5 kg at (0, 2), in metres. Find the centre of mass (X, Y).',
     "parts": ['X (m)', 'Y (m)'],
     "answer": [1.2, 1.0], "tol": [0.005, 0.005],
     "hint": 'X = Σ m x ÷ Σ m, Y = Σ m y ÷ Σ m',
     "working": 'X = (0 + 12 + 0)/10 = 1.2 m; Y = (0 + 0 + 10)/10 = 1.0 m'},
    {"id": 'P2.2', "tier": 2, "group": 'Centre of mass',
     "q": 'Two skaters, 80 kg and 50 kg, stand still and push apart. The 50 kg skater moves off at 2.4 m/s. How fast does the other move (m/s)?',
     "parts": ['speed (m/s)'],
     "answer": [1.5], "tol": [0.005],
     "hint": 'momentum stays 0: m₁ v₁ = m₂ v₂',
     "working": 'Momentum starts at zero and no outside force acts: 80 v = 50 × 2.4 → v = 1.5 m/s the other way; the centre of mass stays where it was'},
    {"id": 'P2.3', "tier": 2, "group": 'The cross product and angular velocity',
     "q": 'P = 4i + j and Q = 2i + 3j. Find P × Q (give the k part).',
     "parts": ['the k part of P × Q'],
     "answer": [10.0], "tol": [0.005],
     "hint": 'P × Q = (Pₓ Q_y − P_y Qₓ) k',
     "working": 'P × Q = (Pₓ Q_y − P_y Qₓ) k = (4 × 3 − 1 × 2) k = 10k'},
    {"id": 'P2.4', "tier": 2, "group": 'Torque and angular momentum',
     "q": 'A 0.4 m wrench is pulled with 60 N at 50° to its handle. What is the torque (N m)?',
     "parts": ['τ (N m)'],
     "answer": [18.385066634855473], "tol": [0.005],
     "hint": 'τ = r F sin θ',
     "working": 'τ = r F sin θ = 0.4 × 60 × sin 50° = 18.39 N m'},
    {"id": 'P2.5', "tier": 2, "group": 'Torque and angular momentum',
     "q": 'A 0.15 kg ball flies at 30 m/s along a straight line that passes 2 m from a post. What is its angular momentum about the post (kg m²/s)?',
     "parts": ['l (kg m²/s)'],
     "answer": [9.0], "tol": [0.005],
     "hint": 'l = m v × lever arm',
     "working": 'l = p × lever arm = 0.15 × 30 × 2 = 9 kg m²/s, the same all along the line'},
    {"id": 'P2.6', "tier": 2, "group": 'Equilibrium',
     "q": 'A 3.0 m plank of 12 kg rests on supports at its two ends. A 50 kg person stands 1.0 m from the left end. Find the upward push of the right support, then the left (N).',
     "parts": ['right support (N)', 'left support (N)'],
     "answer": [222.13333333333335, 385.4666666666667], "tol": [0.005, 0.005],
     "hint": 'torques about one end, then the forces add to 0',
     "working": 'Torques about the left end: R × 3 = (12 × 1.5 + 50 × 1.0) × 9.8 → R = 222.13 N; forces: left = 62 × 9.8 − 222.13 = 385.47 N'},
    {"id": 'P2.7', "tier": 2, "group": 'Moment of inertia',
     "q": 'A uniform disc of 5 kg and radius 0.4 m spins at 20 rad/s about its axis. Find its moment of inertia (kg m²) and its kinetic energy (J).',
     "parts": ['I (kg m²)', 'K (J)'],
     "answer": [0.4, 80.0], "tol": [0.005, 0.005],
     "hint": 'I = ½ M R², K = ½ I ω²',
     "working": 'I = ½ M R² = ½ × 5 × 0.16 = 0.4 kg m²; K = ½ I ω² = ½ × 0.4 × 400 = 80 J'},
    {"id": 'P2.8', "tier": 2, "group": 'Rotational kinematics',
     "q": 'A wheel starts from rest with α = 4 rad/s². After 5 s, what is its angular velocity (rad/s) and the angle turned (rad)?',
     "parts": ['ω (rad/s)', 'θ (rad)'],
     "answer": [20.0, 50.0], "tol": [0.005, 0.005],
     "hint": 'ω = α t, θ = ½ α t²',
     "working": 'ω = 0 + 4 × 5 = 20 rad/s; θ = ½ × 4 × 25 = 50 rad (7.96 turns)'},
    {"id": 'P2.9', "tier": 2, "group": 'Rotational dynamics',
     "q": 'A torque of 8 N m acts on a wheel with I = 0.5 kg m², from rest. Find α (rad/s²) and the time to reach 80 rad/s (s).',
     "parts": ['α (rad/s²)', 't (s)'],
     "answer": [16.0, 5.0], "tol": [0.005, 0.005],
     "hint": 'α = τ ÷ I, t = ω ÷ α',
     "working": 'α = τ/I = 8/0.5 = 16 rad/s²; t = 80/16 = 5 s'},
    {"id": 'P2.10', "tier": 2, "group": 'Angular momentum about an axis',
     "q": 'A skater spins at 2.5 rad/s with I = 2.8 kg m², then pulls in her arms to I = 1.2 kg m². What is her new angular velocity (rad/s)?',
     "parts": ['ω₂ (rad/s)'],
     "answer": [5.833333333333334], "tol": [0.005],
     "hint": 'I₁ ω₁ = I₂ ω₂',
     "working": 'No outside torque: I₁ω₁ = I₂ω₂ → ω₂ = 2.8 × 2.5/1.2 = 5.83 rad/s'},
    {"id": 'P3.1', "tier": 3, "group": 'Centre of mass',
     "q": "The Earth (5.9724 × 10²⁴ kg) and the Moon (7.346 × 10²² kg) are 384,400 km apart. How far from Earth's centre is their centre of mass, to the nearest kilometre? Is it inside Earth (radius 6,371 km)?",
     "parts": ["distance from Earth's centre (km)"],
     "answer": [4670.63808953565], "tol": [0.5],
     "hint": 'X = d × M_Moon ÷ (M_Earth + M_Moon)',
     "working": 'X = 384,400 × 0.07346/(5.9724 + 0.07346) = 4,671 km: inside Earth, about 1,700 km below the surface'},
    {"id": 'P3.2', "tier": 3, "group": 'Torque and angular momentum',
     "q": 'A bolt needs 45 N m to loosen. You can push with 150 N. How long must the spanner be if you push at right angles, and if you can only push at 60° to the handle (m)?',
     "parts": ['at right angles (m)', 'at 60° (m)'],
     "answer": [0.3, 0.34641016151377546], "tol": [0.005, 0.005],
     "hint": 'r = τ ÷ (F sin θ)',
     "working": 'r = τ/(F sin θ): 45/150 = 0.30 m; at 60°: 45/(150 × 0.866) = 0.35 m'},
    {"id": 'P3.3', "tier": 3, "group": 'Equilibrium',
     "q": 'A 1.2 m crowbar rests on a fulcrum 0.15 m from the end under a 1,400 N load. What push at the far end lifts it (N), and what is the mechanical advantage?',
     "parts": ['push (N)', 'mechanical advantage'],
     "answer": [200.0, 7.000000000000001], "tol": [0.005, 0.005],
     "hint": 'load × its arm = push × its arm, both arms from the fulcrum',
     "working": 'Arms are measured from the fulcrum: effort arm 1.2 − 0.15 = 1.05 m. 1,400 × 0.15 = E × 1.05 → E = 200 N; advantage 1,400/200 = 7 (not 1.2/0.15 = 8)'},
    {"id": 'P3.4', "tier": 3, "group": 'Moment of inertia',
     "q": "A ring and a disc, each 2 kg with radius 0.25 m, are each given 30 J of spin. Find each one's angular velocity (rad/s): ring, then disc.",
     "parts": ['ring ω (rad/s)', 'disc ω (rad/s)'],
     "answer": [21.908902300206645, 30.983866769659336], "tol": [0.005, 0.005],
     "hint": '½ I ω² = E, so ω = √(2E ÷ I)',
     "working": 'I ring = 0.125, disc = 0.0625 kg m². ω = √(2E/I): 21.91 and 30.98 rad/s; the disc spins √2 times faster for the same energy'},
    {"id": 'P3.5', "tier": 3, "group": 'Moment of inertia',
     "q": 'A uniform 3 kg rod, 2 m long, turns about one end. Find its moment of inertia (kg m²) and its radius of gyration (m).',
     "parts": ['I (kg m²)', 'k (m)'],
     "answer": [4.0, 1.1547005383792515], "tol": [0.005, 0.005],
     "hint": 'I = M L² ÷ 3, k = √(I ÷ M)',
     "working": 'I = M L²/3 = 3 × 4/3 = 4 kg m²; k = √(4/3) = 1.15 m (= L/√3)'},
    {"id": 'P3.6', "tier": 3, "group": 'Rotational kinematics',
     "q": 'A flywheel slows from 50 rad/s to 30 rad/s while turning 40 times. Find its angular deceleration (rad/s², as a positive number) and how long it took (s).',
     "parts": ['deceleration (rad/s²)', 'time (s)'],
     "answer": [3.183098861837907, 6.283185307179586], "tol": [0.005, 0.005],
     "hint": 'ω² = ω₀² + 2αθ, then t = (ω₀ − ω) ÷ |α|',
     "working": 'θ = 40 × 2π = 251.33 rad; ω² = ω₀² + 2αθ → α = (900 − 2,500)/(2 × 251.33) = -3.18 rad/s²; t = (50 − 30)/3.18 = 6.28 s'},
    {"id": 'P3.7', "tier": 3, "group": 'Rotational dynamics',
     "q": "A 4 kg bucket on a rope wound round a drum of radius 0.15 m falls 3 m in 1.5 s from rest. What is the drum's moment of inertia (kg m²)?",
     "parts": ['I (kg m²)'],
     "answer": [0.24075000000000002], "tol": [0.005],
     "hint": 'a = 2y ÷ t², and a = m g ÷ (m + I ÷ R²)',
     "working": 'a = 2y/t² = 2.6667 m/s²; a = m g/(m + I/R²) → I = R²(m g/a − m) = 0.0225 × (14.7 − 4) = 0.2408'},
    {"id": 'P3.8', "tier": 3, "group": 'Angular momentum about an axis',
     "q": 'A merry-go-round (I = 250 kg m²) turns at 1.0 rad/s. A 30 kg child steps on at the rim, 2 m from the axis. Find the new angular velocity (rad/s) and the kinetic energy lost (J).',
     "parts": ['ω₂ (rad/s)', 'energy lost (J)'],
     "answer": [0.6756756756756757, 40.54054054054055], "tol": [0.005, 0.005],
     "hint": 'I₁ ω₁ = I₂ ω₂, then compare ½ I ω²',
     "working": 'I₂ = 250 + 30 × 4 = 370; ω₂ = 250/370 = 0.6757 rad/s; K: 125 J → 84.46 J, so 40.54 J lost (to friction as the child is dragged up to speed)'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g, w, t) for g, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else f"✓ all {n} right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print(textwrap.fill(f"The formula: {q['hint']}.", 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g, w, t) for g, w, t in zip(got, answers(k), Q[k]["tol"]))

# the checker itself: a right answer passes in any form, a wrong one doesn't
assert right(read_part("0.2 m"), 0.2, 0.005) and right(read_part("2*pi*600/60"), answers("P1.4")[0], 0.005)
assert right(read_part("20*π rad/s"), answers("P1.4")[0], 0.005) and right(read_part("6.53 m/s"), 6.533, 0.005)
assert not right(read_part("0.3"), 0.2, 0.005) and read_part("hello") is None

## Tier 1 · Warm-up

One idea, one step, small numbers: a centre of mass, a cross product's size, revolutions per minute into radians per
second, v = ωr, a torque, a see-saw, a point mass's moment of inertia, ω after a steady α, and power from torque.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Every warm-up is one line. Two habits matter: `math.sin` works in radians, so an angle in degrees goes through
`math.radians` first; and a centre of mass is a weighted average, which numpy writes as `np.average(x, weights=m)`.
Each line below works out one warm-up and checks it against the practice set.

In [ ]:
X = np.average([0, 1.0], weights=[4, 1])                         # P1.1: Σ m x ÷ Σ m
mid = 2.4 / 2                                                    # P1.2: uniform, so the middle
cross = 6 * 2 * math.sin(math.radians(30))                       # P1.3: |A × B| = A B sin θ
w_drill = 600 * 2 * math.pi / 60                                 # P1.4: one turn is 2π radians
v = 10 * 0.4                                                     # P1.5: v = ω r
tau = 0.3 * 50 * math.sin(math.radians(90))                      # P1.6: τ = r F sin θ
m_bal = 20 * 2 / 1                                               # P1.7: m₁ d₁ = m₂ d₂ (g cancels)
I_pt = 3 * 0.5 ** 2                                              # P1.8: I = m r²
w_end = 2 + 3 * 4                                                # P1.9: ω = ω₀ + α t
power = 50 * 100                                                 # P1.10: P = τ ω

print(f"X {X:.3f} m · middle {mid} m · |A × B| {cross:.3f} · drill {w_drill:.2f} rad/s · v {v} m/s · τ {tau:.1f} N m")
print(f"balancing mass {m_bal} kg · I {I_pt} kg m² · ω {w_end} rad/s · P {power} W")
for k, got in (("P1.1", [X]), ("P1.2", [mid]), ("P1.3", [cross]), ("P1.4", [w_drill]), ("P1.5", [v]), ("P1.6", [tau]),
               ("P1.7", [m_bal]), ("P1.8", [I_pt]), ("P1.9", [w_end]), ("P1.10", [power])):
    assert agrees(k, got), k

### Burst the firework

A shell leaves the ground at 30 m/s and 60°. At the top of its arc, 34.44 m up, it bursts in two equal halves: one
stops dead and falls straight down, the other flies on with all the shell's sideways momentum, so at twice the speed.
The burst is all inside forces, so the **centre of mass** (GOLD) carries on along the unbroken shell's arc, and lands
where the shell would have, 79.53 m away. Change the launch, or the share of the mass that stops dead, and watch the
centre of mass keep to its arc.

In [ ]:
def burst(v=30.0, angle=60.0, share=0.5):
    """The shell up to the top of its arc, then the two pieces: a share of the mass stops dead, the rest carries all
    the shell's momentum. Returns the runs and where everything lands."""
    vx, vy = v * math.cos(math.radians(angle)), v * math.sin(math.radians(angle))
    fall = lambda t, s: (s[2], s[3], 0.0, -G)                    # gravity only, no air resistance
    up = simulate(fall, (0, 0, vx, vy), 30, 1e-3, until=(3, 0.0))          # rises until its upward speed is 0
    _, (x_top, h, _, _) = up[-1]
    p1 = simulate(fall, (x_top, h, 0.0, 0.0), 30, 1e-3, until=(1, 0.0))    # stops dead, falls straight down
    p2 = simulate(fall, (x_top, h, vx / (1 - share), 0.0), 30, 1e-3, until=(1, 0.0))   # momentum kept: (1 − share) v' = vx
    whole = simulate(fall, (0, 0, vx, vy), 30, 1e-3, until=(1, 0.0))        # the shell that never burst
    com = share * p1[-1][1][0] + (1 - share) * p2[-1][1][0]
    return up, p1, p2, whole, com

def firework_picture(v=30.0, angle=60.0, share=0.5):
    up, p1, p2, whole, com = burst(v, angle, share)
    fig, ax = plt.subplots(figsize=(11.5, 4.8))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.84, bottom=0.14)
    ax.plot([s[0] for _, s in whole], [s[1] for _, s in whole], color=DIM, ls="--", lw=1.6, label="the unbroken shell")
    ax.plot([s[0] for _, s in up], [s[1] for _, s in up], color=CHALK)
    ax.plot([s[0] for _, s in p1], [s[1] for _, s in p1], color=CHALK, label="the two pieces")
    ax.plot([s[0] for _, s in p2], [s[1] for _, s in p2], color=CHALK)
    cx = [share * a[1][0] + (1 - share) * b[1][0] for a, b in zip(p1, p2)]
    cy = [share * a[1][1] + (1 - share) * b[1][1] for a, b in zip(p1, p2)]
    ax.plot(cx, cy, color=GOLD, ls=":", lw=3, label="their centre of mass")
    ax.scatter([p1[-1][1][0], p2[-1][1][0]], [0, 0], s=70, color=CHALK, zorder=3)
    ax.scatter([com], [0], s=120, color=GOLD, zorder=4)
    ax.set_xlabel("distance (m)"); ax.set_ylabel("height (m)"); ax.set_ylim(bottom=0)
    ax.legend(loc="upper right", frameon=False)
    fig.suptitle(f"pieces land at {p1[-1][1][0]:.2f} m and {p2[-1][1][0]:.2f} m; centre of mass at {com:.2f} m, "
                 f"the shell's own {whole[-1][1][0]:.2f} m", x=0.08, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_, _p1, _p2, _whole, _com = burst()                               # the video's numbers
assert round(_p1[-1][1][0], 2) == 39.77 and round(_p2[-1][1][0], 2) == 119.30 and round(_com, 2) == 79.53
assert close(_com, _whole[-1][1][0]) and close(_com, 30 ** 2 * math.sin(math.radians(120)) / G)
assert close(_p1[-1][0], _p2[-1][0])                              # both pieces land together

In [ ]:
interact(firework_picture, v=FloatSlider(value=30, min=10, max=40, step=1, description="speed (m/s)"),
         angle=FloatSlider(value=60, min=15, max=85, step=1, description="angle (°)"),
         share=FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, description="stops dead"));

## Tier 2 · Core

Several steps each, at the level of the video's worked problems: a centre of mass in two dimensions, skaters pushing
apart, a cross product from components, a torque at an angle, a ball's angular momentum about a post, a plank on two
supports, a disc's inertia and energy, a wheel's spin-up, torque into angular acceleration, and a skater pulling in her
arms.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

**Centres of mass**: `np.average` with the masses as weights, once for x and once for y. **Cross products**: numpy's
`np.cross` on two 3-component vectors gives the whole vector, here only a k part. **Equilibrium**: torques about one
support first (that support's own force has no lever arm there), then the forces. The rest are the rotational
formulas, one line each.

In [ ]:
m21, x21, y21 = [2, 3, 5], [0, 4, 0], [0, 0, 2]                   # P2.1: X and Y separately
assert agrees("P2.1", [np.average(x21, weights=m21), np.average(y21, weights=m21)])
assert agrees("P2.2", [50 * 2.4 / 80])                            # P2.2: momentum stays zero
PxQ = np.cross([4, 1, 0], [2, 3, 0])                              # P2.3: the k part is the last component
print("P × Q =", PxQ, "  so 10k")
assert agrees("P2.3", [PxQ[2]])
assert agrees("P2.4", [0.4 * 60 * math.sin(math.radians(50))])   # P2.4
assert agrees("P2.5", [0.15 * 30 * 2])                            # P2.5: momentum × lever arm
R_right = (12 * 1.5 + 50 * 1.0) * G / 3                           # P2.6: torques about the left end
assert agrees("P2.6", [R_right, (12 + 50) * G - R_right])
I27 = 5 * 0.4 ** 2 / 2                                            # P2.7
assert agrees("P2.7", [I27, I27 * 20 ** 2 / 2])
assert agrees("P2.8", [4 * 5, 4 * 5 ** 2 / 2])                    # P2.8: from rest
a29 = 8 / 0.5                                                     # P2.9: α = τ ÷ I
assert agrees("P2.9", [a29, 80 / a29])
assert agrees("P2.10", [2.8 * 2.5 / 1.2])                         # P2.10: I₁ω₁ = I₂ω₂
print(f"plank: right support {R_right:.2f} N, left {(12 + 50) * G - R_right:.2f} N; disc: {I27} kg m², "
      f"{I27 * 400 / 2:.0f} J; skater: {2.8 * 2.5 / 1.2:.4f} rad/s")

### Pull the puck's cord

A 0.2 kg puck glides on an air table, tied to an elastic cord that runs down through a hole. The cord pulls straight
toward the hole, with a force that grows with the stretch (k r), so it has **no lever arm** about the hole: no torque.
Its angular momentum m v r⊥ must stay put, 0.200 kg m²/s, while the puck swoops in to 0.25 m at 4 m/s, twice as fast
as it started. The dots are equal times apart: far out they're close together (slow), near the hole far apart (fast).
Make the cord stiffer, or flick the puck harder, and watch the line on the right: it never moves.

In [ ]:
def puck_run(k=12.8, v0=2.0, r0=0.5, m=0.2):
    """The puck from (r0, 0) moving sideways at v0, pulled toward the hole by k r, for one full loop."""
    T = 2 * math.pi / math.sqrt(k / m)                            # its path closes after one period
    return simulate(lambda t, s: (s[2], s[3], -k / m * s[0], -k / m * s[1]), (r0, 0.0, 0.0, v0), T, T / 4000)

def puck_picture(k=12.8, v0=2.0):
    m, r0 = 0.2, 0.5
    run = puck_run(k, v0, r0, m)
    t = np.array([t for t, _ in run]); s = np.array([s for _, s in run])
    r = np.hypot(s[:, 0], s[:, 1]); speed = np.hypot(s[:, 2], s[:, 3])
    L = m * (s[:, 0] * s[:, 3] - s[:, 1] * s[:, 2])               # m (x v_y − y v_x) = m v r⊥
    j = int(np.argmin(r))
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.19, wspace=0.28)
    ax.plot(s[:, 0], s[:, 1], color=CHALK, lw=1.6)
    ax.scatter(s[::160, 0], s[::160, 1], s=18, color=CHALK, zorder=3)
    ax.plot([0, s[j, 0]], [0, s[j, 1]], color=CORAL, lw=2)        # the cord's pull, toward the hole
    ax.scatter([0], [0], s=60, color=DIM, zorder=4); ax.scatter([s[j, 0]], [s[j, 1]], s=90, color=CHALK, zorder=5)
    ax.set_aspect("equal"); lim = max(r0, r.max()) * 1.15; ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)"); ax.set_title("the path, dots equal times apart", fontsize=13)
    bx.plot(t, L, color=GOLD, lw=3)
    bx.set_ylim(0, 2 * L[0]); bx.set_xlabel("t (s)"); bx.set_ylabel("m v r⊥ (kg m²/s)")
    bx.set_title("angular momentum about the hole", fontsize=13)
    fig.suptitle(f"closest {r[j]:.3f} m at {speed[j]:.2f} m/s; m v r⊥ stays {L[0]:.3f} kg m²/s "
                 f"(it moved by {L.max() - L.min():.0e})", x=0.07, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_run = puck_run()                                                 # the video's numbers
_r = [math.hypot(s[0], s[1]) for _, s in _run]; _v = [math.hypot(s[2], s[3]) for _, s in _run]
_L = [0.2 * (s[0] * s[3] - s[1] * s[2]) for _, s in _run]
assert close(min(_r), 0.25, 1e-6) and close(max(_v), 4.0, 1e-6) and max(_L) - min(_L) < 1e-9 and close(_L[0], 0.2, 1e-12)

In [ ]:
interact(puck_picture, k=FloatSlider(value=12.8, min=2, max=40, step=0.2, description="cord k (N/m)"),
         v0=FloatSlider(value=2.0, min=0.5, max=4, step=0.1, description="flick (m/s)"));

### Switch off the fan

A fan turning at 30 rad/s is switched off and slows at a steady 1.5 rad/s². The simulation steps it forward until it
stops: 20 s and 300 rad, which is 47.75 turns. On the left, ω falls in a straight line, and the area under it is the
angle turned. On the right, the turns pile up fast at first, then slowly. After 10 s, half the time, it has already
turned 225 rad, three quarters of the way.

In [ ]:
def fan_run(w0=30.0, alpha=-1.5):
    """θ and ω, stepped at a steady α until ω reaches zero."""
    return simulate(lambda t, s: (s[1], alpha), (0.0, w0), 3600, 1e-2, until=(1, 0.0))

def fan_picture(w0=30.0, alpha=-1.5):
    run = fan_run(w0, alpha)
    t = np.array([t for t, _ in run]); th = np.array([s[0] for _, s in run]); w = np.array([s[1] for _, s in run])
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.6))
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.19, wspace=0.25)
    ax.fill_between(t, 0, w, color=SKY, alpha=0.18, lw=0); ax.plot(t, w, color=SKY)
    ax.set_xlabel("t (s)"); ax.set_ylabel("ω (rad/s)"); ax.set_title("ω: the area under it is θ", fontsize=13)
    bx.plot(t, th / (2 * math.pi), color=CHALK); bx.axhline(th[-1] / (2 * math.pi), color=GOLD, ls="--", lw=1.6)
    bx.set_xlabel("t (s)"); bx.set_ylabel("turns"); bx.set_title("the turn counter", fontsize=13)
    fig.suptitle(f"stops after {t[-1]:.2f} s, having turned {th[-1]:.1f} rad = {th[-1] / (2 * math.pi):.2f} turns",
                 x=0.07, ha="left", fontsize=15, color=GOLD)
    footnote(fig); plt.show()

_fan = fan_run()                                                  # the video's numbers
_t10 = [s[0] for t, s in _fan if abs(t - 10) < 5e-3][0]
assert close(_fan[-1][0], 20) and close(_fan[-1][1][0], 300) and close(_t10, 225)
assert round(_fan[-1][1][0] / (2 * math.pi), 2) == 47.75

In [ ]:
interact(fan_picture, w0=FloatSlider(value=30, min=5, max=60, step=1, description="ω₀ (rad/s)"),
         alpha=FloatSlider(value=-1.5, min=-5, max=-0.5, step=0.1, description="α (rad/s²)"));

## Tier 3 · Stretch

Run it backwards, combine ideas, spot the trap: the Earth and Moon's centre of mass, a spanner you can only push at an
angle, a crowbar's arms measured from the wrong place, a ring and a disc given the same energy, a rod about its end, a
flywheel slowing over forty turns, a drum's inertia from a falling bucket, and the energy a merry-go-round loses.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

The stretch problems are the same formulas, rearranged. Then the course's two "direct sums" with numpy: slice a rod
into N pieces and add up m x², and a disc into N rings and add up m r². As N grows the sums close in on the shortcuts,
M L² ÷ 12 and ½ M R²: for a rod, exactly (1 − 1/N²) of the way.

In [ ]:
assert agrees("P3.1", [384400 * 0.07346e24 / (5.9724e24 + 0.07346e24)])            # P3.1: inside Earth
assert agrees("P3.2", [45 / 150, 45 / (150 * math.sin(math.radians(60)))])        # P3.2: r = τ ÷ (F sin θ)
E33 = 1400 * 0.15 / (1.2 - 0.15)                                                   # P3.3: arms from the fulcrum
assert agrees("P3.3", [E33, 1400 / E33])
assert agrees("P3.4", [math.sqrt(2 * 30 / (2 * 0.25 ** 2)), math.sqrt(2 * 30 / (2 * 0.25 ** 2 / 2))])
assert agrees("P3.5", [3 * 2 ** 2 / 3, math.sqrt((3 * 2 ** 2 / 3) / 3)])          # P3.5: about one end
th36 = 40 * 2 * math.pi; a36 = (50 ** 2 - 30 ** 2) / (2 * th36)                     # P3.6: ω² = ω₀² + 2αθ
assert agrees("P3.6", [a36, (50 - 30) / a36])
a37 = 2 * 3 / 1.5 ** 2                                                             # P3.7: a from the fall, then I
assert agrees("P3.7", [0.15 ** 2 * (4 * G / a37 - 4)])
I38 = 250 + 30 * 2 ** 2; w38 = 250 * 1.0 / I38                                     # P3.8: the child steps on
assert agrees("P3.8", [w38, 250 * 1.0 ** 2 / 2 - I38 * w38 ** 2 / 2])
print(f"crowbar: {E33:.0f} N, advantage {1400 / E33:.0f} (not 1.2/0.15 = 8); merry-go-round: {w38:.4f} rad/s, "
      f"{250 / 2 - I38 * w38 ** 2 / 2:.2f} J lost")

def rod_slices(N, M=1.0, L=1.0):
    """Σ (M/N) x² over N equal slices of a rod about its centre, each slice at its own middle."""
    x = (np.arange(N) + 0.5) * L / N - L / 2
    return (M / N * x ** 2).sum()

def disc_rings(N, M=1.0, R=1.0):
    """Σ (ring's mass) r² over N rings of a uniform disc, each ring at its middle radius."""
    i = np.arange(1, N + 1)
    return (M * ((i / N) ** 2 - ((i - 1) / N) ** 2) * ((i - 0.5) * R / N) ** 2).sum()

for N in (2, 10, 100, 10000):
    print(f"N = {N:>6}: rod {rod_slices(N):.6f} (shortcut {1 / 12:.6f})    disc {disc_rings(N):.6f} (shortcut 0.5)")
assert [round(rod_slices(N), 6) for N in (2, 10, 100)] == [0.0625, 0.0825, 0.083325]
assert round(disc_rings(10), 4) == 0.4975 and round(disc_rings(100), 6) == 0.499975
assert all(abs(rod_slices(N) - (1 - 1 / N ** 2) / 12) < 1e-12 for N in (2, 10, 100))

### Drop the bucket

A 5 kg bucket hangs on a rope wound round a solid 20 kg drum of radius 0.2 m. Let go. The simulation treats the rope
as a very stiff spring between the drum's rim and the bucket, and steps both forward: the bucket pulled down by
gravity and up by the rope, the drum turned by the rope's pull at its rim. Nobody tells it the formula, yet after 2 s
the bucket has fallen 6.53 m at 6.53 m/s, just as a = m g ÷ (m + I ÷ R²) says. The energy bars show where the falling
bucket's energy went: a third into the bucket's own motion, two thirds into the spinning drum. Make the drum lighter
or smaller and see the share change.

In [ ]:
def well_run(m=5.0, M=20.0, R=0.2, t_end=2.0, k_rope=2e5, dt=1e-4):
    """The bucket (fallen y, speed v) and the drum (angle φ, ω), joined by a stiff rope; y measured down."""
    I = M * R ** 2 / 2
    def deriv(t, s):
        y, v, ph, w = s
        T = max(0.0, k_rope * (y - R * ph))                       # the rope pulls only when stretched
        return (v, G - T / m, w, T * R / I)
    a = m * G / (m + I / R ** 2)
    s0 = m * (G - a) / k_rope                                     # start the rope at its working stretch
    return simulate(deriv, (s0, 0.0, 0.0, 0.0), t_end, dt), s0, I, a

def well_picture(m=5.0, M=20.0, R=0.2):
    run, s0, I, a = well_run(m, M, R)
    t = np.array([t for t, _ in run]); s = np.array([s for _, s in run])
    y, v, w = s[:, 0] - s0, s[:, 1], s[:, 3]
    kb, kd, pe = m * v[-1] ** 2 / 2, I * w[-1] ** 2 / 2, m * G * y[-1]
    fig, (ax, bx) = plt.subplots(1, 2, figsize=(11.5, 4.6), gridspec_kw={"width_ratios": [1.4, 1]})
    fig.subplots_adjust(left=0.07, right=0.98, top=0.84, bottom=0.23, wspace=0.3)
    ax.plot(t, a * t ** 2 / 2, color=DIM, ls="--", lw=1.6, label="the formula, ½ a t²")
    ax.plot(t[::1000], y[::1000], "o", color=CHALK, label="simulated")
    ax.set_xlabel("t (s)"); ax.set_ylabel("fallen (m)"); ax.legend(loc="upper left", frameon=False)
    labels = ["the motion:\nbucket + drum", "height lost:\nm g y"]
    bx.bar(labels[:1], [kb], color=VIOLET, width=0.55)
    bx.bar(labels[:1], [kd], bottom=[kb], color=VIOLET, alpha=0.5, width=0.55)
    bx.bar(labels[1:], [pe], color=GOLD, width=0.55)
    bx.text(0, kb / 2, "bucket", ha="center", color=SLATE, fontsize=12)
    bx.text(0, kb + kd / 2, "drum", ha="center", color=CHALK, fontsize=12)
    bx.set_ylabel("energy (J)")
    fig.suptitle(f"after 2 s: fallen {y[-1]:.2f} m at {v[-1]:.2f} m/s (a = {a:.2f} m/s²); bucket {kb:.0f} J + drum "
                 f"{kd:.0f} J = {pe:.0f} J", x=0.07, ha="left", fontsize=14, color=GOLD)
    footnote(fig); plt.show()

_run, _s0, _I, _a = well_run()                                   # the video's numbers
_y, _v = _run[-1][1][0] - _s0, _run[-1][1][1]
assert close(_y, _a * 2, 1e-4) and close(_v, _a * 2, 1e-3) and round(_a, 2) == 3.27
assert round(_y, 2) == 6.53 and round(_v, 2) == 6.53 and round(_I * (_run[-1][1][3]) ** 2 / 2) == 213

In [ ]:
interact(well_picture, m=FloatSlider(value=5, min=1, max=20, step=0.5, description="bucket (kg)"),
         M=FloatSlider(value=20, min=2, max=60, step=1, description="drum (kg)"),
         R=FloatSlider(value=0.2, min=0.05, max=0.5, step=0.01, description="radius (m)"));

## Sources
Every exercise and data set here is our own. The Earth's and the Moon's
masses and their distance are NASA's fact sheets (nssdc.gsfc.nasa.gov, retrieved 2026-11-11). The topics follow a
standard first course in rotational motion; the definitions agree with OpenStax, University Physics volume 1, chapters
9.6 (centre of mass), 10 (fixed-axis rotation), 11 (angular momentum) and 12.1 (static equilibrium), retrieved
2026-11-11.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib
_src = pathlib.Path("../../../episodes/ph06_rotation/practice.py")
if _src.exists():
    _spec = importlib.util.spec_from_file_location("ph06_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["tier"]) == (_e["q"], _e["working"], _e["tier"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])), _e["id"]
        assert all(abs(a - float(b)) < 1e-9 for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
        assert list(_q["tol"]) == list(_e["tol"] if isinstance(_e["tol"], tuple) else [_e["tol"]] * len(_q["answer"])), _e["id"]
    assert len(_P.EX) == len(EX)
    print("matches the course's practice set")